# Does preserving each axle improve wheel-profile modeling?
Jamie Le · November 10, 2026

**Question:** Does each axle’s own pair of wheel profiles improve displacement-hunting prediction for a truck, beyond conicity and ordinary wear?

This follows `04_profile_shape_extension.ipynb`. The original three descriptors, coordinate window and kernel model are retained. The change is to preserve axle identity and train on the axle’s own displacement result, then flag a truck if either axle is predicted to hunt. We compare these representations without changing the earlier feature definitions.

The project asks which wheel characteristics accompany hunting. This experiment tests whether a physically relevant representation helps identify that information. It complements teammates’ wear/conicity, rail-profile and algorithm work. Previous ablations, profile cases, rail interactions and PCA are already complete and are not repeated here as new discoveries.

## Reproduction

We use this notebook for our profile-model comparisons in the report and presentation. Install Python 3.12 and the packages in [requirements.txt](../requirements.txt), following [INSTALL.md](../INSTALL.md). We need only the base packages; this notebook does not require TensorFlow, XGBoost or MATLAB.

Get the data through the private **-MxV-Wheel-Profile** course Teams channel: **Shared → Documents → Data**. Download `Profiles/`, `Eq_Con_Fit_Results.xlsm` and `final_merged_data.csv`, preserving the original folder structure. These profile experiments read:

```text
data/
├── final_merged_data.csv
└── Profiles/
    ├── Candidate wheel profiles/Car*_Axle*[LR].WHL
    └── WNTs/2023-06-21 - Hunt_Disp_Max_CP_Difference.csv
```

We expect all 112 measured WHL files, separate from the unworn reference. We also accept `final_merged_data(in).csv`. If you keep the downloaded folder outside the repository, set its location before starting Jupyter:

macOS/Linux:
```bash
export HUNTING_DATA_DIR="/full/path/to/Data"
```

Windows PowerShell:
```powershell
$env:HUNTING_DATA_DIR = "C:\full\path\to\Data"
```

The default is the repository's `data/` folder. A locally rebuilt `outputs/data/final_merged_data.csv` takes precedence; regenerate or remove it when changing input releases.

From the repository root, activate the environment and start `python -m notebook`. Open this notebook and choose **Restart Kernel and Run All Cells**. We include the experiment functions below, so no unpublished helper modules, previous notebook session or historical-result files are required. The existing `src/contact_hunting.py` and `src/profile_shapes.py` supply the shared kernel and WHL readers.

We export figures as 300 dpi PNGs and vector PDFs and keep the tables behind them under the local output folder named below. The model runs recompute settings and predictions, using complete-car folds and 2,000 bootstrap draws with seed 42. To inspect a completed run without retraining, open its local tables or saved executed notebook. Those results apply to that run's inputs and code; rerun after either changes.

Keep partner-derived tables, images and executed notebooks in the private Teams storage. The Git notebook has cleared outputs. Confirm instructor access to the private data and any saved intermediates before submission.

**Output folder:** `outputs/axle_model/`.


In [ ]:
from pathlib import Path
import os
import sys
import hashlib
import json
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

project_folder = next((p for p in (Path.cwd(), *Path.cwd().parents)
                       if (p / "src" / "contact_hunting.py").is_file()), None)
if project_folder is None:
    raise RuntimeError("Start Jupyter in the repository root or notebooks/; see INSTALL.md.")
sys.path.insert(0, str(project_folder / "src"))
configured_data = Path(os.environ.get("HUNTING_DATA_DIR", "data")).expanduser()
data_folder = configured_data if configured_data.is_absolute() else project_folder / configured_data

def merged_path(root):
    paths = [Path(root) / "outputs/data/final_merged_data.csv",
             data_folder / "final_merged_data.csv", data_folder / "final_merged_data(in).csv"]
    for path in paths:
        if path.is_file():
            return path
    raise FileNotFoundError("Missing merged CSV. See this notebook's input instructions and INSTALL.md.")

folder = project_folder / "outputs" / "axle_model"
folder.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False, 'figure.dpi': 120})


## Experiment functions

We first define own-axle features, the fixed OR rule and complete-car evaluation. The next cell prepares the common inputs and compares the four feature bundles. Source data remain unchanged.

In [ ]:
"""Axle-own-geometry experiment with complete-car nested evaluation.

The training label belongs to an axle. Model settings are selected on inner
truck balanced accuracy after a fixed OR rule; native axle performance is a
separate descriptive result. Decision scores are not probabilities.
"""

from pathlib import Path
import sys

import numpy as np
import pandas as pd

from contact_hunting import (
    DISP_SPEED, PARAMETER_GRID, RAIL_PROFILES, SPEED_LIMIT,
    classification_metrics, kernel_scores,
)
from profile_shapes import SHAPE_COLUMNS, TREAD_GRID, describe_tread


TRUCK_KEY = ["Car", "Truck", "Rail_Profile"]
AXLE_KEY = ["Car", "Truck", "Axle", "Rail_Profile"]
WEAR_SOURCES = {
    "wear_flange_width": ["Flange Width_L", "Flange Width_R"],
    "wear_flange_height": ["Flange Height (Sh)_L", "Flange Height (Sh)_R"],
    "wear_abs_hollow": ["|Max Hollow Wear - AAR|_L", "|Max Hollow Wear - AAR|_R"],
    "wear_hollow_position": ["Max Hollow Wear Position - AAR_L", "Max Hollow Wear Position - AAR_R"],
}


def _checked_keys(frame):
    missing = set(AXLE_KEY).difference(frame.columns)
    if missing or frame.empty:
        raise ValueError("Axle assessments need nonempty complete identifiers.")
    ids = frame[AXLE_KEY].to_numpy(dtype=float)
    if not np.isfinite(ids).all() or (ids != np.floor(ids)).any():
        raise ValueError("Car, truck, axle and rail identifiers must be finite integers.")
    if (frame.Car <= 0).any() or not frame.Axle.isin([1, 2, 3, 4]).all():
        raise ValueError("Unsupported measured car or axle identifier.")
    if not np.array_equal(frame.Truck, np.where(frame.Axle <= 2, 1, 2)):
        raise ValueError("Truck identifiers disagree with their axle mapping.")
    if not frame.Rail_Profile.isin(RAIL_PROFILES).all():
        raise ValueError("Unsupported rail identifier.")
    if frame.duplicated(AXLE_KEY).any():
        raise ValueError("Duplicate axle/rail assessments.")
    for (_, truck, _), group in frame.groupby(TRUCK_KEY, sort=True):
        expected = {1, 2} if truck == 1 else {3, 4}
        if len(group) != 2 or set(group.Axle) != expected:
            raise ValueError("Each truck/rail assessment requires its two correct axles.")


def prepare_axles(merged, wheels, profiles):
    """Make own-axle features without reading or modifying source files.

    Wear is averaged over the two sides of the same axle. Source rail rows
    never change the weight assigned to a wheel. Shape definitions use the
    existing centered tread window, whose physical interpretation is provisional.
    """
    _checked_keys(merged)
    sources = [column for pair in WEAR_SOURCES.values() for column in pair]
    needed = ["Eq Con 3mm", DISP_SPEED, *sources]
    if not set(needed).issubset(merged):
        raise ValueError("Required axle conicity, displacement or wear columns are missing.")
    numeric = merged[needed].to_numpy(dtype=float)
    if not np.isfinite(numeric).all():
        raise ValueError("Axle measurements must be finite.")
    if ((merged[DISP_SPEED] <= 0) | (merged[DISP_SPEED] > SPEED_LIMIT)).any():
        raise ValueError("Axle displacement speeds must be in (0, 80].")
    if (merged.groupby(["Car", "Axle"])[sources].nunique() != 1).any().any():
        raise ValueError("Axle wear measurements must be invariant across rail rows.")

    wheel_columns = {"Car", "Axle", "Side", "tread_slope", "tread_nonlinearity"}
    if not wheel_columns.issubset(wheels):
        raise ValueError("Required own-wheel descriptors are missing.")
    if wheels.duplicated(["Car", "Axle", "Side"]).any():
        raise ValueError("Duplicate wheel descriptors.")
    expected = {(int(car), int(axle), side)
                for car, axle in merged[["Car", "Axle"]].drop_duplicates().itertuples(index=False)
                for side in ("L", "R")}
    wheel_keys = set(wheels[["Car", "Axle", "Side"]].itertuples(index=False, name=None))
    if wheel_keys != expected or set(profiles) != expected:
        raise ValueError("Own-wheel and profile coverage must match the measured axles.")
    if not np.isfinite(wheels[["tread_slope", "tread_nonlinearity"]].to_numpy(float)).all():
        raise ValueError("Own-wheel descriptors must be finite.")
    wheel_index = wheels.set_index(["Car", "Axle", "Side"]).sort_index()
    shape_rows = []
    for car, axle in sorted({(c, a) for c, a, _ in expected}):
        for side in ("L", "R"):
            slope, nonlinear = describe_tread(profiles[(car, axle, side)])
            supplied = wheel_index.loc[(car, axle, side)]
            if not np.allclose([slope, nonlinear],
                               [supplied.tread_slope, supplied.tread_nonlinearity],
                               rtol=1e-10, atol=1e-12):
                raise ValueError("Wheel descriptors disagree with their own profile.")
        own = wheel_index.loc[(car, axle)]
        difference = np.asarray(profiles[(car, axle, "L")]) - profiles[(car, axle, "R")]
        shape_rows.append({
            "Car": car, "Axle": axle,
            SHAPE_COLUMNS[0]: own.tread_slope.mean(),
            SHAPE_COLUMNS[1]: own.tread_nonlinearity.mean(),
            # Keep the original name so feature bundles can match. Here there
            # is one axle, so no maximum across the truck's two axles is taken.
            SHAPE_COLUMNS[2]: float(np.sqrt(np.mean(difference ** 2))),
        })
    result = merged[AXLE_KEY + ["Eq Con 3mm", DISP_SPEED] + sources].copy()
    result["conicity_mean"] = result["Eq Con 3mm"]
    result["displacement_hunted"] = (result[DISP_SPEED] < SPEED_LIMIT).astype(int)
    for name, pair in WEAR_SOURCES.items():
        result[name] = result[pair].mean(axis=1)
    retained = AXLE_KEY + ["displacement_hunted", "conicity_mean", *WEAR_SOURCES]
    result = result[retained].merge(pd.DataFrame(shape_rows), on=["Car", "Axle"],
                                     how="left", validate="many_to_one")
    return result.sort_values(AXLE_KEY).reset_index(drop=True)


def aggregate_scores(frame, scores):
    """Return true OR label and fixed max-score OR prediction per truck/rail."""
    _checked_keys(frame)
    values = np.asarray(scores, dtype=float)
    if values.shape != (len(frame),) or not np.isfinite(values).all():
        raise ValueError("One finite signed score is needed per axle assessment.")
    if "displacement_hunted" not in frame or not frame.displacement_hunted.isin([0, 1]).all():
        raise ValueError("Axle displacement labels must be binary.")
    paired = frame[AXLE_KEY + ["displacement_hunted"]].copy()
    paired["score"] = values
    trucks = paired.groupby(TRUCK_KEY, as_index=False, sort=True).agg(
        actual=("displacement_hunted", "max"), score=("score", "max"))
    trucks["predicted"] = (trucks.score >= 0).astype(int)
    return trucks


def evaluate_axle_models(frame, feature_sets, parameter_grid=PARAMETER_GRID):
    """Nested leave-car-out axle training, tuned using pooled inner truck BA.

    Returns wide axle predictions (including raw scores), wide truck predictions
    after OR, and selected settings. These tables contain partner-derived data
    and remain local. The same numeric grid fixes search budget, not identical
    effective regularization to truck training with half as many rows.
    """
    _checked_keys(frame)
    if "displacement_hunted" not in frame or not frame.displacement_hunted.isin([0, 1]).all():
        raise ValueError("Axle displacement labels must be binary.")
    frame = frame.reset_index(drop=True)
    car_ids = frame.Car.to_numpy()
    cars = np.sort(np.unique(car_ids))
    if len(cars) < 4:
        raise ValueError("At least four complete car groups are required.")
    y = frame.displacement_hunted.to_numpy(dtype=int)
    if set(np.unique(y)) != {0, 1}:
        raise ValueError("Both displacement classes are required.")
    allowed = {"conicity_mean", *WEAR_SOURCES, *SHAPE_COLUMNS}
    reserved = set(AXLE_KEY + ["actual", "score", "predicted"])
    if not feature_sets or any(name in reserved or name.endswith("__score") for name in feature_sets):
        raise ValueError("Feature-set names must be nonempty and preserve output columns.")
    if any(not set(columns).issubset(allowed) for columns in feature_sets.values()):
        raise ValueError("Only own-axle geometry and wear predictors are allowed.")
    grid = tuple(parameter_grid)
    if not grid or any(len(pair) != 2 or not np.isfinite(pair).all() or min(pair) <= 0 for pair in grid):
        raise ValueError("Kernel settings must be finite positive alpha/gamma pairs.")
    features = {name: frame[columns].to_numpy(dtype=float) for name, columns in feature_sets.items()}
    if any(not np.isfinite(x).all() for x in features.values()):
        raise ValueError("Model features must be finite.")
    rails = frame.Rail_Profile.to_numpy()
    axle_predictions = frame[AXLE_KEY].copy()
    axle_predictions["actual"] = y
    truck_predictions = aggregate_scores(frame, np.zeros(len(frame)))[TRUCK_KEY + ["actual"]]
    settings = []

    def score_model(name, fit, test, alpha, gamma):
        if set(car_ids[fit]) & set(car_ids[test]):
            raise AssertionError("Training and validation/test cars overlap.")
        if set(np.unique(y[fit])) != {0, 1}:
            raise ValueError("A training fold has only one displacement class.")
        return kernel_scores(features[name][fit], features[name][test],
                             rails[fit], rails[test], y[fit], alpha, gamma)

    for name in feature_sets:
        outer_scores = np.full(len(frame), np.nan)
        seen = np.zeros(len(frame), dtype=int)
        for held_car in cars:
            train = np.flatnonzero(car_ids != held_car)
            test = np.flatnonzero(car_ids == held_car)
            inner_folds = np.array_split(cars[cars != held_car], 3)
            candidates = []
            for alpha, gamma in grid:
                inner_scores = np.full(len(frame), np.nan)
                inner_seen = np.zeros(len(frame), dtype=int)
                for validation_cars in inner_folds:
                    validation = train[np.isin(car_ids[train], validation_cars)]
                    fit = train[~np.isin(car_ids[train], validation_cars)]
                    inner_scores[validation] = score_model(name, fit, validation, alpha, gamma)
                    inner_seen[validation] += 1
                if not np.all(inner_seen[train] == 1) or np.any(inner_seen[test]):
                    raise AssertionError("Inner validation did not cover training cars exactly once.")
                inner_trucks = aggregate_scores(frame.iloc[train], inner_scores[train])
                ba = classification_metrics(inner_trucks.actual, inner_trucks.predicted)["balanced_accuracy"]
                candidates.append((ba, float(alpha), float(gamma)))
            ba, alpha, gamma = max(candidates, key=lambda item: (item[0], item[1], -item[2]))
            outer_scores[test] = score_model(name, train, test, alpha, gamma)
            seen[test] += 1
            settings.append(dict(held_car=int(held_car), model=name, alpha=alpha, gamma=gamma,
                                 inner_truck_balanced_accuracy=ba))
        if not np.all(seen == 1) or not np.isfinite(outer_scores).all():
            raise AssertionError("Each axle assessment must receive one finite held-out score.")
        axle_predictions[name] = (outer_scores >= 0).astype(int)
        axle_predictions[name + "__score"] = outer_scores
        aggregate = aggregate_scores(frame, outer_scores)
        pd.testing.assert_frame_equal(truck_predictions[TRUCK_KEY + ["actual"]],
                                      aggregate[TRUCK_KEY + ["actual"]])
        truck_predictions[name] = aggregate.predicted.to_numpy()
    return axle_predictions, truck_predictions, pd.DataFrame(settings)


In [ ]:
"""Fixed local axle-versus-truck experiment. Source inputs are read only."""
from pathlib import Path
import hashlib
import json
import os
import platform
import sys

import numpy as np
import pandas as pd

ROOT = project_folder
DATA = data_folder
from contact_hunting import (load_tables, prepare_trucks, evaluate_models,
                             classification_metrics, paired_car_intervals)
from profile_shapes import prepare_truck_shapes, SHAPE_COLUMNS

FEATURE_SETS = {
    "Conicity": ["conicity_mean"],
    "Conicity + shapes": ["conicity_mean", *SHAPE_COLUMNS],
    "Wear": ["conicity_mean", *WEAR_SOURCES],
    "Wear + shapes": ["conicity_mean", *WEAR_SOURCES, *SHAPE_COLUMNS],
}
KEY = ["Car", "Truck", "Rail_Profile"]
OLD_NAMES = {
    "Conicity": "Mean conicity + rail",
    "Conicity + shapes": "Mean conicity + shape + rail",
    "Wear": "Wear + conicity + rail",
    "Wear + shapes": "Wear + conicity + all shapes + rail",
}
PREVIOUS = Path(os.environ["HUNTING_REFERENCE_DIR"]) if os.environ.get("HUNTING_REFERENCE_DIR") else None


def hashes(paths, relative_to):
    return {str(p.relative_to(relative_to)) if p.is_relative_to(relative_to) else p.name: hashlib.sha256(p.read_bytes()).hexdigest()
            for p in sorted(paths)}


def protected_state():
    """Record input and tracked-code hashes, excluding generated outputs."""
    sources = [merged_path(ROOT), DATA / "Profiles/WNTs/2023-06-21 - Hunt_Disp_Max_CP_Difference.csv"]
    sources += list((DATA / "Profiles/Candidate wheel profiles").glob("*.WHL"))
    code = list((ROOT / "src").glob("*.py"))
    code += list((ROOT / "notebooks").glob("*.ipynb"))
    code += list((ROOT / "tests").glob("*.py"))
    return {"source_hashes": hashes(sources, ROOT), "code_hashes": hashes(code, ROOT)}


def load_inputs():
    merged = pd.read_csv(merged_path(ROOT))
    contact = pd.read_csv(DATA / "Profiles/WNTs/2023-06-21 - Hunt_Disp_Max_CP_Difference.csv", encoding="cp1252")
    trucks = prepare_trucks(merged, contact)
    shapes, wheels, profiles = prepare_truck_shapes(DATA / "Profiles/Candidate wheel profiles", trucks)
    trucks = trucks.merge(shapes, on=["Car", "Truck"], validate="many_to_one")
    axles = prepare_axles(merged, wheels, profiles)
    wear = axles.drop_duplicates(["Car", "Axle"]).groupby(["Car", "Truck"], as_index=False)[list(WEAR_SOURCES)].mean()
    trucks = trucks.merge(wear, on=["Car", "Truck"], validate="many_to_one")
    if PREVIOUS is not None:
        prior = pd.read_csv(PREVIOUS / "truck_features.csv")
        pd.testing.assert_frame_equal(trucks.sort_values(KEY).reset_index(drop=True),
                                      prior.sort_values(KEY).reset_index(drop=True),
                                      check_exact=False, rtol=1e-12)
    actual = aggregate_scores(axles, axles.displacement_hunted.to_numpy(float) - .5)
    check = trucks.merge(actual, on=KEY, validate="one_to_one")
    assert np.array_equal(check.displacement_hunted, check.actual)
    assert len(axles) == 392 and len(trucks) == 196 and trucks.Car.nunique() == 14
    return trucks, axles, wheels, profiles


def metrics_row(scenario, unit, model, actual, predicted):
    return dict(scenario=scenario, unit=unit, model=model,
                observations=len(actual), **classification_metrics(actual, predicted))


def check_prior(predictions, scenario):
    if PREVIOUS is None:
        return
    old = pd.read_csv(PREVIOUS / f"{scenario}_displacement_hunted_predictions.csv")
    joined = predictions.merge(old, on=KEY, validate="one_to_one", suffixes=("_new", "_old"))
    assert len(joined) == len(predictions) == len(old)
    assert np.array_equal(joined.actual_new, joined.actual_old)
    for new, prior in OLD_NAMES.items():
        assert np.array_equal(joined[new], joined[prior]), (scenario, new)


def run_experiment(output_dir=None):
    before = protected_state()
    trucks, axles, wheels, profiles = load_inputs()
    metrics, changes, predictions, settings, diagnostics = [], [], [], [], []
    census = axles.groupby(KEY).displacement_hunted.sum().rename("positive_axles").reset_index()
    for row in census.itertuples(index=False):
        diagnostics.append(dict(kind="census", scenario="all_cars", Car=row.Car,
                                Truck=row.Truck, Rail_Profile=row.Rail_Profile,
                                positive_axles=row.positive_axles))
    for scenario in ("all_cars", "without_car_7"):
        t = trucks if scenario == "all_cars" else trucks[trucks.Car != 7].reset_index(drop=True)
        a = axles if scenario == "all_cars" else axles[axles.Car != 7].reset_index(drop=True)
        _, truck_pred, truck_settings = evaluate_models(t, "displacement_hunted", FEATURE_SETS)
        check_prior(truck_pred, scenario)
        print(f"{scenario}: summary-model predictions computed", flush=True)
        axle_pred, pooled_pred, axle_settings = evaluate_axle_models(a, FEATURE_SETS)
        assert np.array_equal(pooled_pred[KEY].to_numpy(), truck_pred[KEY].to_numpy())
        assert np.array_equal(pooled_pred.actual, truck_pred.actual)
        joined = truck_pred[KEY + ["actual"]].copy()
        for model in FEATURE_SETS:
            joined["Truck: " + model] = truck_pred[model]
            joined["Axle OR: " + model] = pooled_pred[model]
            for unit, pred in (("truck_original", truck_pred), ("truck_axle_or", pooled_pred),
                               ("axle_native", axle_pred)):
                metrics.append(metrics_row(scenario, unit, model, pred.actual, pred[model]))
                keys = KEY if unit != "axle_native" else ["Car", "Truck", "Axle", "Rail_Profile"]
                long = pred[keys + ["actual"]].copy()
                long["predicted"] = pred[model]
                if unit == "axle_native":
                    long["score"] = pred[model + "__score"]
                elif unit == "truck_axle_or":
                    long["score"] = aggregate_scores(a, axle_pred[model + "__score"].to_numpy()).score
                long["scenario"], long["unit"], long["model"] = scenario, unit, model
                predictions.append(long)
        comparisons = []
        for prefix in ("Truck: ", "Axle OR: "):
            comparisons += [(prefix + "Conicity + shapes", prefix + "Conicity", prefix + "shape beyond conicity"),
                            (prefix + "Wear + shapes", prefix + "Wear", prefix + "shape beyond wear")]
        comparisons += [("Axle OR: " + m, "Truck: " + m, "Axle OR versus truck: " + m) for m in FEATURE_SETS]
        _, delta = paired_car_intervals(joined, comparisons=comparisons, n_bootstrap=2000, seed=42)
        delta["scenario"] = scenario
        changes.append(delta)
        for unit, chosen in (("truck_original", truck_settings), ("axle_to_truck", axle_settings)):
            chosen = chosen.copy()
            chosen["scenario"], chosen["unit"] = scenario, unit
            settings.append(chosen)
        labels = a.groupby(KEY).displacement_hunted.sum().rename("positive_axles").reset_index()
        for model in joined.columns.difference(KEY + ["actual"]):
            z = joined[KEY + ["actual", model]].merge(labels, on=KEY, validate="one_to_one")
            for count, g in z.groupby("positive_axles"):
                diagnostics.append(dict(kind="stratum", scenario=scenario, model=model,
                                        positive_axles=int(count), observations=len(g),
                                        correct=int((g.actual == g[model]).sum()),
                                        false_positive=int(((g.actual == 0) & (g[model] == 1)).sum()),
                                        false_negative=int(((g.actual == 1) & (g[model] == 0)).sum())))
        for model in FEATURE_SETS:
            for car, g in joined.groupby("Car"):
                old_errors = int((g["Truck: " + model] != g.actual).sum())
                new_errors = int((g["Axle OR: " + model] != g.actual).sum())
                diagnostics.append(dict(kind="per_car", scenario=scenario, model=model, Car=int(car),
                                        observations=len(g), original_errors=old_errors,
                                        axle_or_errors=new_errors, error_reduction=old_errors-new_errors))
        print(f"{scenario}: axle models completed", flush=True)
    after = protected_state()
    assert before == after, "An original input, source notebook or implementation changed"
    result = dict(metrics=pd.DataFrame(metrics), comparisons=pd.concat(changes, ignore_index=True),
                  predictions=pd.concat(predictions, ignore_index=True),
                  settings=pd.concat(settings, ignore_index=True), diagnostics=pd.DataFrame(diagnostics))
    verification = dict(originals_unchanged=True, historical_predictions_checked=PREVIOUS is not None,
                        protected_state=before, python=platform.python_version(), numpy=np.__version__,
                        pandas=pd.__version__, feature_sets=FEATURE_SETS,
                        tuning_objective="truck balanced accuracy after fixed OR in inner car folds",
                        native_axle_metric="secondary; settings selected for truck endpoint",
                        bootstrap_draws=2000, seed=42,
                        note="Repeated exploratory 14-car sample; no calibrated probabilities, causal rule or new-car validation.")
    if output_dir is not None:
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        for name, frame in result.items():
            frame.to_csv(output_dir / f"{name}.csv", index=False)
        (output_dir / "verification.json").write_text(json.dumps(verification, indent=2) + "\n")
    result["verification"] = verification
    return result


## Comparison design

Four fixed feature bundles enter both representations: conicity; conicity + three shapes; conicity + four wear means; and conicity + wear + shapes. Rail identity enters every model. The labels “Wear” and “Wear + shapes” also include conicity and rail. No new window, model family or threshold search is added.

We check how often the two axles have different outcomes before interpreting label mixing. We also compare false alarms with hunting recall, since taking an OR can trade specificity for recall. We measure both below.

Each axle uses its own equivalent conicity, mean signed fitted slope across its two wheels, mean RMS departure from fitted lines, and left/right RMS profile distance. Its wear features are means of its own two supplied wheel values. In the saved axle table, `conicity_mean` is the own-axle conicity and `max_left_right_shape_difference` is the own-axle RMS, with no maximum across axles. The original truck model uses four-wheel means and the larger left/right distance of the two axles. All curves use x=60–110, centered at x=85. Units, alignment and export provenance remain provisional.

One complete car is held out at a time. Three inner car folds select from the original six kernel settings, using **truck balanced accuracy after fixed OR aggregation**. Scaling uses fitting rows only. The zero decision threshold is unchanged. Native axle scores are secondary: settings were selected for the truck endpoint. Kernel scores are not probabilities. Saved continuous scores are available for the new axle and OR models; original-truck score fields are blank because the existing comparator returns binary decisions.

Both scenarios are fixed in advance: all cars and a complete rerun without Car 7, retaining its known source anomaly in the main experiment. We recompute the four original truck models in each scenario. Input and source-code hashes are checked before and after execution. Historical prediction checks run only when HUNTING_REFERENCE_DIR is set; the verification file records whether we ran them.

This changes representation, training labels, row count and the OR decision together. It cannot isolate the effect of averaging alone. The same regularization grid means the same search budget, not identical effective shrinkage with twice as many rows. Acceleration retains its previous primary exploratory status; the new experiment is about displacement only.

Methods: [nested validation](https://scikit-learn.org/stable/auto_examples/model_selection/plot_nested_cross_validation_iris.html), [nonoverlapping group folds](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupKFold.html).

In [ ]:
# Fresh training and evaluation; results stay in outputs/.
# No source files or Git objects are changed.
result = run_experiment(output_dir=folder)
metrics = result['metrics']
comparisons = result['comparisons']
predictions = result['predictions']
diagnostics = result['diagnostics']
trucks, axles, wheels, profiles = load_inputs()
assert result['verification']['originals_unchanged']
display(pd.Series({'Cars': axles.Car.nunique(), 'Wheel profiles': len(wheels),
                   'Axles': len(axles[['Car','Axle']].drop_duplicates()),
                   'Axle/rail rows': len(axles), 'Truck/rail rows': len(trucks)}, name='Coverage'))

## Does it improve the original truck endpoint?

Balanced accuracy averages hunting recall and no-hunting recall. Compare models within each feature bundle; the extended bundles are not a search from which to select a favorable winner. Ordinary accuracy is retained in the saved metrics but does not replace balanced accuracy.

In [ ]:
main = metrics[(metrics.scenario=='all_cars') & (metrics.unit!='axle_native')].copy()
view = main.set_index(['model','unit'])[['balanced_accuracy','recall','specificity']]
display((100*view).round(1).rename(columns={'balanced_accuracy':'Balanced accuracy (%)',
       'recall':'Hunting recall (%)','specificity':'No-hunting recall (%)'}))
fig, axes = plt.subplots(1,3,figsize=(14,4),sharey=True)
x = np.arange(len(FEATURE_SETS))
for ax, metric, title in zip(axes,['balanced_accuracy','recall','specificity'],
                            ['Balanced accuracy','Hunting recall','No-hunting recall']):
    for offset,unit,color,label in [(-.19,'truck_original','#7e8c98','Original truck'),
                                   (.19,'truck_axle_or','#226584','Own axle → truck OR')]:
        y=main[main.unit==unit].set_index('model').loc[list(FEATURE_SETS),metric]
        ax.bar(x+offset,100*y,width=.36,color=color,label=label)
    ax.set_xticks(x,['Conicity','Conicity+shape','Wear','Wear+shape'])
    ax.set(title=title,ylim=(0,100));ax.grid(axis='y',alpha=.2);ax.set_axisbelow(True)
axes[0].set_ylabel('Percent');axes[0].legend(fontsize=9)
fig.suptitle('Held-out complete cars · displacement outcome · 196 truck/rail assessments')
fig.tight_layout();fig.savefig(folder / 'axle_review_performance.png', dpi=300); fig.savefig(folder / 'axle_review_performance.pdf', bbox_inches='tight')
plt.show()

## Added information and uncertainty

The ranges use 2,000 paired resamples of entire cars from the fixed held-out predictions. They omit retraining, exploratory feature discovery and repeated use of these same 14 cars. They are descriptive ranges, with no adjustment for the several planned comparisons. A range crossing zero leaves improvement uncertain.

In [ ]:
changes = comparisons[comparisons.scenario=='all_cars'].copy()
display(changes.assign(**{c:100*changes[c] for c in
    ['change_in_balanced_accuracy','lower_95','upper_95']})
    [['comparison','change_in_balanced_accuracy','lower_95','upper_95',
      'cars_with_fewer_errors','cars_with_more_errors','cars_tied']].round(1))
fig,ax=plt.subplots(figsize=(11,5.5))
y=np.arange(len(changes))
lo=100*changes.lower_95.to_numpy();hi=100*changes.upper_95.to_numpy()
point=100*changes.change_in_balanced_accuracy.to_numpy()
ax.hlines(y,lo,hi,color='#226584',linewidth=2)
ax.scatter(point,y,color='#226584',zorder=3)
ax.axvline(0,color='#777777',linewidth=1)
ax.set_yticks(y,changes.comparison);ax.invert_yaxis()
ax.set_xlabel('Change in balanced accuracy (percentage points)')
ax.set_title('Paired complete-car ranges · fixed held-out predictions')
ax.grid(axis='x',alpha=.15);fig.tight_layout()
fig.savefig(folder / 'axle_review_changes.png', dpi=300); fig.savefig(folder / 'axle_review_changes.pdf', bbox_inches='tight');plt.show()

## Axle results and the label-agreement limit

Native axle scores evaluate each axle’s own displacement label. They are a different endpoint from truck balanced accuracy and use settings selected for the truck. The row count doubles without increasing the 14 independent cars.

In [ ]:
native=metrics[(metrics.scenario=='all_cars') & (metrics.unit=='axle_native')]
display((100*native.set_index('model')[['balanced_accuracy','recall','specificity']]).round(1))
census=axles.groupby(KEY).displacement_hunted.sum().rename('Positive axles').reset_index()
display(census['Positive axles'].value_counts().reindex([0,1,2],fill_value=0).rename('Truck/rail cases').to_frame())
strata=diagnostics[(diagnostics.kind=='stratum') & (diagnostics.scenario=='all_cars')].copy()
strata['Correct (%)']=100*strata.correct/strata.observations
display(strata[['model','positive_axles','observations','correct','false_positive','false_negative','Correct (%)']].round(1))

## Every discordant case and one fixed geometry example

All assessments with exactly one displacement-positive axle are shown. They repeat source cars and trucks. The plotted example is the lexicographically first Car/Truck/Rail among them, selected by the declared rule, not by a favorable shape-model prediction. The other axle’s profile is visible beside it. Acceleration, conicity and wear remain competing context; no profile difference is treated as an isolated cause.

In [ ]:
discordant_keys=census.loc[census['Positive axles']==1,KEY]
discordant=axles.merge(discordant_keys,on=KEY,validate='many_to_one')
display(discordant[KEY+['Axle','displacement_hunted','conicity_mean',*SHAPE_COLUMNS]].round(4))
example=discordant_keys.sort_values(KEY).iloc[0]
car,truck,rail=(int(example[c]) for c in KEY)
example_axles=discordant[(discordant.Car==car)&(discordant.Truck==truck)&(discordant.Rail_Profile==rail)]
fig,axes=plt.subplots(1,2,figsize=(11,4.5),sharex=True,sharey=True)
for axis,row in zip(axes,example_axles.sort_values('Axle').itertuples()):
    for side,color in [('L','#226584'),('R','#c87432')]:
        axis.plot(TREAD_GRID,profiles[(car,row.Axle,side)],label=side,color=color)
    axis.set_title(f'Axle {row.Axle}: displacement label {row.displacement_hunted}')
    axis.set_xlabel('Export lateral coordinate (units unconfirmed)');axis.grid(alpha=.2);axis.legend()
axes[0].set_ylabel('Height relative to x=85 (units unconfirmed)')
fig.suptitle(f'First discordant case: Car {car}, Truck {truck}, Rail {rail}')
fig.tight_layout();fig.savefig(folder / 'axle_review_profiles.png', dpi=300); fig.savefig(folder / 'axle_review_profiles.pdf', bbox_inches='tight');plt.show()
display(example_axles[KEY+['Axle','conicity_mean',*list(FEATURE_SETS['Wear'])[1:]]].round(4))
case_predictions=predictions[(predictions.scenario=='all_cars') &
    (predictions.Car==car)&(predictions.Truck==truck)&(predictions.Rail_Profile==rail)]
display(case_predictions[['unit','model','Axle','actual','predicted','score']].round(4))
display(trucks[(trucks.Car==car)&(trucks.Truck==truck)&(trucks.Rail_Profile==rail)]
        [KEY+['acceleration_hunted','displacement_hunted']])
# Audit whether correct truck decisions also identify the positive axle.
localized=predictions[(predictions.scenario=='all_cars') & (predictions.unit=='axle_native')].merge(
    discordant_keys,on=KEY,validate='many_to_one')
local_rows=[]
for (model,c,t,r),g in localized.groupby(['model',*KEY]):
    pos=int(g.loc[g.actual==1,'predicted'].iloc[0])
    sister=int(g.loc[g.actual==0,'predicted'].iloc[0])
    local_rows.append({'model':model,'Car':c,'Truck':t,'Rail_Profile':r,
        'Positive axle detected':pos,'False sister flag':sister,
        'Truck detected':int(pos or sister),'Truck correct only via wrong axle':int(not pos and sister),
        'Both axle labels correct':int(pos and not sister)})
localization=pd.DataFrame(local_rows)
display(localization.groupby('model')[['Positive axle detected','False sister flag','Truck detected',
        'Truck correct only via wrong axle','Both axle labels correct']].sum())
display(localized[localized.model=='Conicity + shapes'][KEY+['Axle','actual','predicted','score']].round(4))


## Which cars benefit, and does the Car 7 sensitivity change the result?

The full car census reports errors removed by the axle OR pipeline: positive values mean fewer errors and negative values mean more errors. All cars are shown. This does not attribute a correction to a specific descriptor.

The Car 7 sensitivity retrains the full pipeline with that car absent. It is a changed sample, not a confidence interval or a reason to discard the main result.

In [ ]:
percar=diagnostics[(diagnostics.kind=='per_car')&(diagnostics.scenario=='all_cars')]
display(percar.pivot(index='Car',columns='model',values='error_reduction').astype(int))
sensitivity=metrics[(metrics.scenario=='without_car_7')&(metrics.unit!='axle_native')]
display((100*sensitivity.set_index(['model','unit'])[['balanced_accuracy','recall','specificity']]).round(1))
display(comparisons[comparisons.scenario=='without_car_7'].assign(
    gain_pp=lambda d:100*d.change_in_balanced_accuracy,
    lower_pp=lambda d:100*d.lower_95,upper_pp=lambda d:100*d.upper_95)
    [['comparison','gain_pp','lower_pp','upper_pp']].round(1))

## Interpreting the comparison

We compare the complete feature bundles using balanced accuracy, hunting recall and no-hunting recall together. The paired ranges and per-car checks show whether a change is consistent across the sample or concentrated in a few cars. The standard-wear comparison tests whether the three WHL descriptors add information beyond ordinary measurements.

For the axle-OR experiment, we also inspect false sister-axle flags and whether a correct truck prediction identified the positive axle. For the paired experiment, the model predicts the truck outcome and cannot identify an unstable axle. These are different modeling questions.

We keep favorable and unfavorable results in the generated tables. The same cars have informed repeated feature development, so the results remain exploratory. Coordinate units, orientation and reconstruction also limit physical interpretation. We need new cars to test transfer and controlled profile changes to investigate effects within the simulator. The scores do not establish causes or maintenance thresholds.


In [ ]:
# Integrity evidence, not a claim of independent new-car validation.
assert result['verification']['protected_state']==protected_state()
assert not predictions.duplicated(['scenario','unit','model','Car','Truck','Axle','Rail_Profile']).any()
for row in metrics.itertuples():
    p=predictions[(predictions.scenario==row.scenario)&(predictions.unit==row.unit)&(predictions.model==row.model)]
    direct=.5*sum((p.loc[p.actual==k,'predicted']==k).mean() for k in [0,1])
    assert np.isclose(direct,row.balanced_accuracy)
print('Verified: source inputs and code unchanged; balanced accuracies independently checked.')